In [ ]:
from tools.utils import *
from time import sleep
from tqdm import tqdm
from ollama import Client
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import torch
import spacy
import json

# Use ChatGPT as Judge to resolve disagreements between models

In [ ]:
clause_type = 'modification' # 'modification' |'opt-out'  |'arbitration' | 'opt-out' | 'class_waiver' | 'anti-scraping'


In [ ]:
# Set up OpenAI API credentials
from openai import OpenAI
openai_client = OpenAI()

In [ ]:
df = pd.read_csv(f'annotations/ollama_annotations/{clause_type}_annotations.csv')
df.shape

In [ ]:
df['agreement'] = True
df['mean'] = df[[c for c in df.columns if c.startswith('label_')]].mean(axis=1)
df.loc[df['mean'].between(0.1,.9), 'agreement'] = False
df['agreement'].value_counts()

In [ ]:

def generate_response_openai(prompt,client,model='gpt-4o'):
    
    response = client.responses.create(
        model=model,
        input=[{"role": "user", "content": prompt}],
        
        )

    return response.output[0].content[0].text.strip().lower()



model = "gpt-4o"
ollama_model = "gpt-oss:120b"
tqdm.pandas()
df[f'response_{model}'] = df.progress_apply(
            lambda row: generate_response_openai( row['prompt'].strip(),
                                                  client = openai_client, 
                                                  model=model) if not row['agreement'] else row[f'label_{ollama_model}'], axis=1
                                            )   


In [ ]:
df[f'label_{model}'] = df[f'response_{model}'].apply(lambda x: 1 if x in ['yes', 'yes.',1] else 0 if x in ['no', 'no.',0] else 0)
df[f'label_{model}'].value_counts(dropna=False)


In [ ]:
df[['sentence', 'sum'] + [c for c in df.columns if c.startswith('label_')]].to_csv(f'annotations/ollama_annotations/{clause_type}_labels_judged.csv')
df.to_csv(f'annotations/ollama_annotations/{clause_type}_annotations_judged.csv')

# Fin.